## 2. Quality Assurance Report

### 2.1 Objective and Methodology

The purpose of this quality-assurance report is to evaluate the completeness, consistency, and technical integrity of the UrbanSound8K dataset before processing it through our APAC compression pipeline.

We perform automated checks on the dataset metadata and WAV audio files to identify missing values, invalid ranges, duplicate entries, and corrupted or unreadable recordings.

We also inspect audio properties such as sampling rates, channel counts, data types, and durations to identify variations that could affect subsequent signal processing.

The checks are performed on the available local dataset. The number of inspected records and audio files will be reported to distinguish the full UrbanSound8K collection from any smaller test subset.

These tests identify detectable data-quality problems but do not guarantee that the audio recordings or their sound-class labels are semantically correct.

The results are summarized in a quality-assurance table, with detected problems and limitations documented for reproducibility.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import soundfile as sf
import hashlib

# Dataset location
DATASET_DIR = Path(
    r"C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\archive"
)

# UrbanSound8K folder structure
METADATA_PATH = DATASET_DIR / "UrbanSound8K.csv"
AUDIO_DIR = DATASET_DIR

print("Dataset found:", DATASET_DIR.exists())
print("Metadata found:", METADATA_PATH.exists())
print("Audio folder found:", AUDIO_DIR.exists())


Dataset found: True
Metadata found: True
Audio folder found: True


In [2]:

# Load metadata
metadata = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata.shape)
print("\nColumns:")
print(metadata.columns.tolist())

print("\nFirst five rows:")
display(metadata.head())

# Count missing values in each column
missing_values = metadata.isna().sum()

print("\nMissing values per column:")
display(missing_values.to_frame(name="Missing count"))

print("\nTotal missing values:", int(missing_values.sum()))


Metadata shape: (8732, 8)

Columns:
['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold', 'classID', 'class']

First five rows:


,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing



Missing values per column:


,Missing count
slice_file_name,0
fsID,0
start,0
end,0
salience,0
fold,0
classID,0
class,0



Total missing values: 0


In [3]:

# Check whether every file listed in the CSV exists
required_columns = {"slice_file_name", "fold"}

missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

def get_audio_path(row):
    return (
        AUDIO_DIR
        / f"fold{int(row['fold'])}"
        / str(row["slice_file_name"])
    )

missing_audio = []

for _, row in metadata.dropna(
    subset=["slice_file_name", "fold"]
).iterrows():

    path = get_audio_path(row)

    if not path.is_file():
        missing_audio.append(str(path))

print("Metadata records:", len(metadata))
print("Missing referenced audio files:", len(missing_audio))

if missing_audio:
    print("\nExamples of missing files:")
    for path in missing_audio[:10]:
        print(path)


Metadata records: 8732
Missing referenced audio files: 0


In [4]:

from collections import Counter

# Find all WAV files physically present
actual_files = {
    p.relative_to(AUDIO_DIR).as_posix()
    for p in AUDIO_DIR.rglob("*.wav")
}

# Get all expected file paths from metadata
expected_paths = [
    f"fold{int(row['fold'])}/{row['slice_file_name']}"
    for _, row in metadata.iterrows()
]

expected_files = set(expected_paths)

# Compare metadata and disk
missing_files = expected_files - actual_files
extra_files = actual_files - expected_files

# Check duplicate references in metadata
duplicate_references = sum(
    count - 1
    for count in Counter(expected_paths).values()
    if count > 1
)

print("=== DATASET COMPLETENESS ===")
print("Metadata records:", len(metadata))
print("Expected unique WAV files:", len(expected_files))
print("Actual WAV files:", len(actual_files))
print("Missing WAV files:", len(missing_files))
print("Extra WAV files:", len(extra_files))
print("Duplicate metadata file references:", duplicate_references)

if missing_files:
    print("\nMissing:", sorted(missing_files)[:10])

if extra_files:
    print("\nExtra:", sorted(extra_files)[:10])


=== DATASET COMPLETENESS ===
Metadata records: 8732
Expected unique WAV files: 8732
Actual WAV files: 8732
Missing WAV files: 0
Extra WAV files: 0
Duplicate metadata file references: 0


In [6]:

import soundfile as sf

corrupt_files = []
checked_files = 0

for relative_path in sorted(expected_files):
    path = AUDIO_DIR / relative_path

    try:
        # Read WAV metadata and decode audio
        info = sf.info(path)
        data, sr = sf.read(
            path,
            dtype="float32",
            always_2d=True
        )

        if (
            info.frames <= 0
            or info.channels <= 0
            or sr <= 0
            or data.shape != (info.frames, info.channels)
            or not np.isfinite(data).all()
        ):
            corrupt_files.append(
                (relative_path, "Invalid audio properties")
            )

        checked_files += 1

    except Exception as e:
        corrupt_files.append((relative_path, str(e)))

print("=== AUDIO READABILITY ===")
print("Files examined:", checked_files)
print("Unreadable or invalid files:", len(corrupt_files))

for path, reason in corrupt_files[:10]:
    print(path, ":", reason)


=== AUDIO READABILITY ===
Files examined: 8732
Unreadable or invalid files: 0


### 2.2 Metadata Completeness and Audio Readability

We evaluated the completeness and readability of the UrbanSound8K dataset using automated Python checks.

The metadata CSV contains 8,732 records across eight columns. No missing values were detected in any of the metadata fields.

We compared the audio files referenced by the metadata against the files physically present in the dataset directory. All 8,732 expected WAV files were found, with no missing files, extra files, or duplicate file references.

We also performed an audio readability test using the Python `soundfile` library. Each WAV file was opened and decoded, and its dimensions, sampling rate, channel count, and sample values were checked for basic validity.

| Quality Check | Result |
|---|---|
| Metadata records | 8,732 |
| Missing metadata values | 0 |
| Missing referenced audio files | 0 |
| Extra WAV files | 0 |
| Duplicate metadata file references | 0 |
| Unreadable or invalid audio files | 0 |

**Conclusion**

The dataset passed all implemented metadata completeness and audio readability checks. Every expected audio file is present and can be decoded successfully.

However, these checks do not guarantee that the recordings are correctly labeled, contain no duplicate audio content, or match the original dataset release byte-for-byte.

Further checks will examine valid data ranges, duplicate audio content, and file integrity.